# scikit-learn N-gram 실습 (초보자용)

**환경:** Google Colab / Jupyter Notebook


## 오늘 배울 것
BoW(단어 개수 세기)는 **단어 순서를 모릅니다.** 그래서 아래 두 리뷰가 똑같이 보입니다.

```
"배송 빠르다 포장 별로"   →  배송은 좋고, 포장이 별로
"포장 빠르다 배송 별로"   →  포장은 좋고, 배송이 별로
```

**N-gram**은 붙어 나온 단어를 **묶음으로 세어서** 순서 정보를 일부 되살립니다.

## 실습 진행 과정
| 단계 | 할 일 | 확인할 결과 |
|---|---|---|
| 0. 준비 | 데이터와 예제 리뷰 불러오기 | 리뷰 수 |
| 1. N-gram 개념 | 직접 묶음 만들어 보기 | 1단어·2단어·3단어 묶음 |
| 2. CountVectorizer로 N-gram | `ngram_range` 바꿔 보기 | 순서만 다른 두 리뷰가 구별됨 |
| 3. 실제 데이터 | 자주 붙어 나온 2단어 묶음 찾기 | 상위 묶음 15개 |
| 4. 칸 수 늘어나는 문제 | `min_df`로 줄이기 | 칸 수 비교표 |
| 5. 정리 | 한 줄 정리, 연습 문제 | — |

> **실행 방법:** 셀을 클릭하고 `Shift + Enter`. 반드시 **위에서부터 순서대로** 실행하세요.

---
## 0. 준비
- 예제 리뷰로 진행합니다.
- `mini`는 개념을 확인할 때 쓰는 **예제 리뷰 5개**입니다.

| 번호 | mini 리뷰 | 눈여겨볼 점 |
|---|---|---|
| 0 | 배송 빠르다 포장 꼼꼼하다 | 칭찬 리뷰 |
| 1 | 배송 빠르다 포장 별로 | 1번과 2번은 **단어는 같고 순서만 다름** |
| 2 | 포장 빠르다 배송 별로 | (뜻은 서로 다름) |
| 3 | 가격 저렴하다 품질 좋다 | 3번과 4번은 **뜻이 반대** |
| 4 | 가격 비싸다 품질 안 좋다 | (`좋다` vs `안 좋다`) |

In [4]:
# 파일 찾기 도구
import glob
# 표 다루기 도구
import pandas as pd

# 개념 확인용 예제 리뷰 5개 (단원 2처럼 단어가 나뉜 모양)
mini = [
    "배송 빠르다 포장 꼼꼼하다",
    "배송 빠르다 포장 별로",
    "포장 빠르다 배송 별로",
    "가격 저렴하다 품질 좋다",
    "가격 비싸다 품질 안 좋다",
]

docs = mini * 20

# 공통 설정: 띄어쓰기로만 단어 나누기, 영어 대소문자 그대로 두기
opts = dict(tokenizer=str.split, token_pattern=None, lowercase=False)

> 💡 **`opts`는 무엇인가요?** 이미 단어를 나눴으니 "띄어쓰기로만 나눠 주세요"라고 알려 주는 설정입니다.
> 이 설정이 없으면 scikit-learn이 **한 글자 단어(`안`, `향`)를 지워** 버립니다. 뒤에서 `**opts`라고 쓰면 이 설정을 그대로 넣는다는 뜻입니다.

---
## 1. N-gram 개념

### 💡 비유: 낱말 카드 묶기
`안`과 `좋다`를 따로 세면 부정의 뜻이 사라집니다. **`안 좋다`를 한 장의 카드**로 만들면 뜻이 남습니다.
이렇게 **연속된 N개 단어 묶음**을 N-gram이라고 합니다.

| 이름 | 묶는 단어 수 | 예: `품질 안 좋다` |
|---|---|---|
| 유니그램 (1-gram) | 1개 | `품질`, `안`, `좋다` |
| 바이그램 (2-gram) | 2개 | `품질 안`, `안 좋다` |
| 트라이그램 (3-gram) | 3개 | `품질 안 좋다` |

직접 만들어 봅니다.

In [5]:
# 문장과 묶을 단어 수(n)를 받아 N-gram 목록을 돌려주는 함수
def make_ngrams(text, n):
    words = text.split()                                              # 띄어쓰기로 단어 나누기
    return [" ".join(words[i:i + n]) for i in range(len(words) - n + 1)]  # i번째부터 n개씩 묶기

# 예제 문장
text = "가격 비싸다 품질 안 좋다"

# 1개, 2개, 3개씩 묶어 보기
print("1-gram:", make_ngrams(text, 1))
print("2-gram:", make_ngrams(text, 2))
print("3-gram:", make_ngrams(text, 3))

1-gram: ['가격', '비싸다', '품질', '안', '좋다']
2-gram: ['가격 비싸다', '비싸다 품질', '품질 안', '안 좋다']
3-gram: ['가격 비싸다 품질', '비싸다 품질 안', '품질 안 좋다']


✅ 2-gram에 **`안 좋다`** 가 생겼습니다. 단어 하나씩 볼 때는 없던 정보입니다.

---
## 2. CountVectorizer로 N-gram 만들기 (7분)
scikit-learn에서는 `ngram_range=(최소, 최대)`만 정하면 됩니다.

| 설정 | 뜻 |
|---|---|
| `ngram_range=(1, 1)` | 1단어만 (기본값, BoW와 같음) |
| `ngram_range=(1, 2)` | 1단어 + 2단어 묶음 |
| `ngram_range=(2, 2)` | 2단어 묶음만 |

### 2-1. 1단어만 셀 때

In [12]:
# 단어 개수 세기 도구
from sklearn.feature_extraction.text import CountVectorizer

# 개념 확인용 예제 리뷰 5개 (단원 2처럼 단어가 나뉜 모양)
mini = [
    "배송 빠르다 포장 꼼꼼하다",
    "배송 빠르다 포장 별로",
    "포장 빠르다 배송 별로",
    "가격 저렴하다 품질 좋다",
    "가격 비싸다 품질 안 좋다",
]
docs = mini * 20
# 공통 설정: 띄어쓰기로만 단어 나누기, 영어 대소문자 그대로 두기
opts = dict(tokenizer=str.split, token_pattern=None, lowercase=False)

# 1단어만 세기
cv1 = CountVectorizer(ngram_range=(1, 1), **opts)
X1 = cv1.fit_transform(mini)

# 리뷰 × 단어 개수 표
pd.DataFrame(X1.toarray(), columns=cv1.get_feature_names_out(), index=mini)

,가격,꼼꼼하다,배송,별로,비싸다,빠르다,안,저렴하다,좋다,포장,품질
배송 빠르다 포장 꼼꼼하다,0,1,1,0,0,1,0,0,0,1,0
배송 빠르다 포장 별로,0,0,1,1,0,1,0,0,0,1,0
포장 빠르다 배송 별로,0,0,1,1,0,1,0,0,0,1,0
가격 저렴하다 품질 좋다,1,0,0,0,0,0,0,1,1,0,1
가격 비싸다 품질 안 좋다,1,0,0,0,1,0,1,0,1,0,1


In [7]:
# 1번과 2번 리뷰가 같은 숫자 목록인지 확인
print(mini[1], "→", X1[1].toarray()[0])
print(mini[2], "→", X1[2].toarray()[0])
print("같은가?", (X1[1].toarray() == X1[2].toarray()).all())

배송 빠르다 포장 별로 → [0 0 1 1 0 1 0 0 0 1 0]
포장 빠르다 배송 별로 → [0 0 1 1 0 1 0 0 0 1 0]
같은가? True


✅ 뜻이 다른데 **완전히 같습니다.** 1단어만 세면 순서를 모르기 때문입니다.

### 2-2. 1단어 + 2단어 묶음을 셀 때

In [13]:
# 1단어 + 2단어 묶음 세기
# 개념 확인용 예제 리뷰 5개 (단원 2처럼 단어가 나뉜 모양)
mini = [
    "배송 빠르다 포장 꼼꼼하다",
    "배송 빠르다 포장 별로",
    "포장 빠르다 배송 별로",
    "가격 저렴하다 품질 좋다",
    "가격 비싸다 품질 안 좋다",
]

# 공통 설정: 띄어쓰기로만 단어 나누기, 영어 대소문자 그대로 두기
opts = dict(tokenizer=str.split, token_pattern=None, lowercase=False)

cv2 = CountVectorizer(ngram_range=(1, 2), **opts)
X2 = cv2.fit_transform(mini)

# 칸 수 비교
print("1단어만 칸 수:", X1.shape[1])
print("1~2단어 칸 수:", X2.shape[1])

# 리뷰 × 묶음 개수 표
pd.DataFrame(X2.toarray(), columns=cv2.get_feature_names_out(), index=mini)

1단어만 칸 수: 11
1~2단어 칸 수: 25


,가격,가격 비싸다,가격 저렴하다,꼼꼼하다,배송,배송 별로,배송 빠르다,별로,비싸다,비싸다 품질,...,저렴하다,저렴하다 품질,좋다,포장,포장 꼼꼼하다,포장 별로,포장 빠르다,품질,품질 안,품질 좋다
배송 빠르다 포장 꼼꼼하다,0,0,0,1,1,0,1,0,0,0,...,0,0,0,1,1,0,0,0,0,0
배송 빠르다 포장 별로,0,0,0,0,1,0,1,1,0,0,...,0,0,0,1,0,1,0,0,0,0
포장 빠르다 배송 별로,0,0,0,0,1,1,0,1,0,0,...,0,0,0,1,0,0,1,0,0,0
가격 저렴하다 품질 좋다,1,0,1,0,0,0,0,0,0,0,...,1,1,1,0,0,0,0,1,0,1
가격 비싸다 품질 안 좋다,1,1,0,0,0,0,0,0,1,1,...,0,0,1,0,0,0,0,1,1,0


In [14]:
# 1번과 2번 리뷰가 이제 다른지 확인
print("같은가?", (X2[1].toarray() == X2[2].toarray()).all())

# 1번 리뷰에만 있는 묶음과 2번 리뷰에만 있는 묶음
names = cv2.get_feature_names_out()
only1 = [n for n, a, b in zip(names, X2[1].toarray()[0], X2[2].toarray()[0]) if a > b]
only2 = [n for n, a, b in zip(names, X2[1].toarray()[0], X2[2].toarray()[0]) if b > a]
print("1번에만:", only1)
print("2번에만:", only2)

같은가? False
1번에만: ['배송 빠르다', '빠르다 포장', '포장 별로']
2번에만: ['배송 별로', '빠르다 배송', '포장 빠르다']


✅ `배송 빠르다`와 `포장 빠르다`처럼 **묶음이 달라서** 이제 두 리뷰가 구별됩니다.
✅ 4번 리뷰 줄을 보면 **`안 좋다`** 칸이 있습니다. 3번 리뷰(`품질 좋다`)와 구별할 단서가 생겼습니다.

---
## 3. 실제 데이터에서 자주 붙어 나온 단어

In [15]:
# 2단어 묶음만 세기
# 개념 확인용 예제 리뷰 5개 (단원 2처럼 단어가 나뉜 모양)
mini = [
    "배송 빠르다 포장 꼼꼼하다",
    "배송 빠르다 포장 별로",
    "포장 빠르다 배송 별로",
    "가격 저렴하다 품질 좋다",
    "가격 비싸다 품질 안 좋다",
]

# 공통 설정: 띄어쓰기로만 단어 나누기, 영어 대소문자 그대로 두기
opts = dict(tokenizer=str.split, token_pattern=None, lowercase=False)

cv_bi = CountVectorizer(ngram_range=(2, 2), **opts)
X_bi = cv_bi.fit_transform(docs)

# 묶음별 전체 횟수 구하기 (세로로 더하기)
bigram = pd.Series(X_bi.sum(axis=0).A1, index=cv_bi.get_feature_names_out())

# 많이 나온 순서로 15개
bigram.sort_values(ascending=False).head(15)

,0
배송 빠르다,40
빠르다 포장,40
가격 저렴하다,20
배송 별로,20
비싸다 품질,20
가격 비싸다,20
빠르다 배송,20
안 좋다,20
저렴하다 품질,20
포장 꼼꼼하다,20


### ✏️ 특정 단어로 시작하는 묶음 찾기
`start_word`를 바꿔 보세요. (예: `안`, `배송`, `가격`, `화질`)

In [16]:
# ===== ✏️ [입력] 시작 단어 =====
start_word = "안"
# ==============================

# start_word로 시작하는 묶음만 골라 많이 나온 순서로 10개
found = bigram[bigram.index.str.startswith(start_word + " ")]
found.sort_values(ascending=False).head(10)

,0
안 좋다,20


🤔 **생각해 볼 점:** 단원 2에서 조사·어미를 지웠기 때문에, 원래 문장에서는 떨어져 있던 단어가 붙어서 묶음이 되기도 합니다.
예: "배송**이** 빨라**서** 좋아요" → `배송 빠르다`, `빠르다 좋다`
의미 있는 묶음인지, 전처리 때문에 생긴 묶음인지는 **원래 리뷰와 비교해서** 판단해야 합니다.

---
## 4. 칸 수가 늘어나는 문제와 `min_df`
묶는 단어 수를 늘리면 **묶음 종류(칸 수)** 가 크게 늘어납니다. 칸이 많으면 메모리를 많이 쓰고, 한두 번만 나온 묶음은 분석에 도움이 되지 않습니다.
`min_df=5`는 **"5개 미만의 리뷰에만 나온 묶음은 버린다"** 는 설정입니다.

In [17]:
# 결과를 담을 목록
rows = []

# 묶음 범위 3가지 × min_df 2가지를 모두 시험
for ngram in [(1, 1), (1, 2), (1, 3)]:
    for min_df in [1, 5]:
        cv = CountVectorizer(ngram_range=ngram, min_df=min_df, **opts).fit(docs)   # 사전 만들기
        rows.append({"묶음 범위": str(ngram), "min_df": min_df, "칸 수": len(cv.vocabulary_)})

# 표로 보기
pd.DataFrame(rows)

,묶음 범위,min_df,칸 수
0,"(1, 1)",1,11
1,"(1, 1)",5,11
2,"(1, 2)",1,25
3,"(1, 2)",5,25
4,"(1, 3)",1,35
5,"(1, 3)",5,35


✅ 묶음 범위를 넓히면 칸 수가 **몇 배씩** 늘고, `min_df`를 쓰면 크게 줄어듭니다.
보통 `ngram_range=(1, 2)`와 `min_df`를 **함께** 씁니다. 알맞은 값은 데이터 크기에 따라 다르므로 직접 비교해 정합니다.

---
## 5. 정리

### ✅ 한 줄 정리
**N-gram = 붙어 나온 단어를 묶음으로 세기.** 순서 정보가 일부 살아나지만 칸 수가 크게 늘어납니다.

| 장점 | 단점 |
|---|---|
| `안 좋다`처럼 부정 표현을 살림 | 칸 수가 크게 늘어남 |
| 순서만 다른 리뷰를 구별 | 멀리 떨어진 단어 관계는 여전히 모름 |

### ✏️ 연습 문제
**문제 1.** `mini`로 `ngram_range=(2, 2)`(2단어 묶음만)를 만들면 칸 수는 몇 개인가요?
**문제 2.** 실제 데이터에서 `ngram_range=(1, 2), min_df=10`일 때 칸 수를 구하세요.

In [ ]:
# 문제 1: 여기에 코드 작성

In [ ]:
# 문제 2: 여기에 코드 작성

<details><summary><b>정답 보기</b> (클릭)</summary>

```python
# 문제 1
cv = CountVectorizer(ngram_range=(2, 2), **opts).fit(mini)
print(len(cv.vocabulary_))

# 문제 2
cv = CountVectorizer(ngram_range=(1, 2), min_df=10, **opts).fit(docs)
print(len(cv.vocabulary_))
```
</details>



**참고 공식 문서:** [scikit-learn CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html)